ok... we need a function for backtracking, which assumes we have a solution and checks it until the solution fails. 
reads in Q1 and Q2, our checking function, potential solution f, and seen

In [1]:
# can be used by strict and semi isomorphism checks too!!!

def backtrack(Q1, Q2, f, seen, check):
    
    if len(f) == len(Q1): # potential solution works! never includes final pair so add to f
        leftover1 = [q for q in Q1 if q not in f]
        leftover2 = [q for q in Q2 if q not in f.values()]
        f[leftover1[0]] = leftover2[0]
        return True

    q1 = Q1[len(f)] # get next value in Q1

    for q2 in Q2:
        f[q1] = q2 # get mapping of Q2 from Q1
        
        if check(f): # check the extention to the next state
            seen.add(q2)

            if backtrack(Q1, Q2, f, seen, check): # move to next state - v cyclical kinda bad?
                return True
                
            seen.remove(q2) # potential solution failed! retreat
        f.pop(q1)
    return False

In [2]:
from itertools import permutations

def is_weak_isomorphism(A1, A2):
    
    def weak_check(f): # check if t2(f(q),g(a)) == f(t1(q,a)) 
        for q1, q2 in f.items():
            for a1 in S1:
                a2 = g[a1]
                q1_next = t1[(q1,a1)]
                if q1_next in f:
                    q2_next = t2[(q2,a2)]
                    if f[q1_next] != q2_next:
                        return False
        return True

    Q1, S1, t1, qs1, F1 = A1
    Q2, S2, t2, qs2, F2 = A2
    
    S1 = list(S1) # annoyed i cant set variables as lists in the lines above
    S2 = list(S2)
    F1 = list(F1)
    F2 = list(F2)

    if len(Q1) != len(Q2) or len(S1) != len(S2) or len(F1) != len(F2):
        return False, None, None

    for trial in permutations(S2): # trying different orders of S2 until one permutation matches
        g = dict(zip(S1, trial)) # set g = {S1 : trial}
        f = {qs1: qs2} # assuming f(qs1) == qs2 !!
        seen = {qs2}

        # since we have already used qs1 and qs2, we can remove them from Q1 and Q2
        Q1n = [q for q in Q1 if q != qs1]
        Q2n = [q for q in Q2 if q != qs2]
        
        # backtrack search to see if f and g are solutions
        if backtrack(Q1n, Q2n, f, seen, weak_check):
            for end1 in F1: # checking f(F1) == F2
                if f[end1] not in F2:
                    return False, None, None
            return True, f, g

    return False, None, None

In [3]:
A1 = (
    {0,1,2,3,4},
    {0,1},
    {   (0,0):1, (0,1):4,
        (1,0):0, (1,1):3,
        (2,0):3, (2,1):0,
        (3,0):2, (3,1):1,
        (4,0):1, (4,1):2},
    1,
    {3,2})

A2 = (
    {4,5,6,7,8},
    {3,2},
    {   (4,3):5, (4,2):8,
        (5,3):4, (5,2):7,
        (6,3):7, (6,2):4,
        (7,3):6, (7,2):5,
        (8,3):5, (8,2):6},
    5, 
    {7,6})

is_weak_isomorphism(A1, A2)

(True, {1: 5, 2: 6, 3: 7, 4: 8, 0: 4}, {0: 3, 1: 2})

In [4]:
# function to display A1 and A2 side by side graphically and backtrack search to show matching
# and also show different structure if not isomorphic?

from IPython.display import display
from search_examples import backtrack_search_frames, interact_traversal_widgets

def visualise(A1, A2, iso_type):
    result, f, g  = iso_type(A1, A2)
    if result:

        G1 = nx.Graph()
        G2 = nx.Graph()
        frames = depth_first_search_frames(G, 0)
        
        i_slider, graph = interact_traversal_widgets(G, frames)
        display(i_slider)
        display(graph)

ModuleNotFoundError: No module named 'search_examples'